# Soup Kernel Challenge: RMSNorm experiment
Experimental RMSNorm branch. The validated baseline is saved as tag t4-validated-v2.
Set COMMIT to its actual commit SHA for submission. GPU T4 x2, Internet on.
Only GPU 0 is used by the unchanged harness.
The clone cell refuses to overwrite an existing repo. Preserve edits and outputs before starting a fresh session.
The measured run includes two repeats and a separate profiling run. Tests and checks stop on errors.
The development repository is public; no GITHUB_TOKEN is required.


In [ ]:
REPO_URL = "https://github.com/samenov01/soup-kernel-lab.git"
COMMIT = "rmsnorm-experiment"  # Pin to the experiment SHA before a measured run.


In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
subprocess.run(["python", "-m", "pip", "install", "-q", "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0", "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest"], check=True)


In [ ]:
import base64, os, shutil, subprocess
from pathlib import Path
if Path("/kaggle/working/repo").exists():
    raise RuntimeError("repo already exists. Preserve any edits and outputs, then use a fresh Kaggle session.")
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
from pathlib import Path
assert Path("/kaggle/working/repo/solution/vocab.py").exists(), "Check REPO_URL and COMMIT: use soup-kernel-lab and its candidate commit."
subprocess.run(["python", "task/prepare_data.py"], cwd="/kaggle/working/repo", check=True)
subprocess.run(["python", "-m", "pytest", "solution", "-q"], cwd="/kaggle/working/repo", check=True)
subprocess.run(["python", "-m", "pytest", "tests", "-q"], cwd="/kaggle/working/repo", env=dict(os.environ, SOUP_APPLY_SOLUTION="1"), check=True)


In [ ]:
subprocess.run(["python", "harness/run_task.py", "--repeats", "2", "--profile", "--out", "runs_rmsnorm_01"], cwd="/kaggle/working/repo", check=True)


In [ ]:
subprocess.run(["python", "harness/check_rules.py", "--result", "runs_rmsnorm_01/result.json"], cwd="/kaggle/working/repo", check=True)
subprocess.run(["python", "harness/score.py", "runs_rmsnorm_01/result.json"], cwd="/kaggle/working/repo", check=True)


While iterating, a quicker run of your solution alone (checks 1 and 5-8 need the base, so they fail
in this mode): `python harness/run_task.py --only solution --out runs_try`. Add `--profile` to get a
table of every CUDA kernel's GPU time in `runs/solution_profile/profile.json`. Logs of every run
are in `runs/<run>/train.log`.